In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["diagnosis"] = data.target

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

In [ ]:
df["diagnosis"].value_counts()

In [ ]:
sns.countplot(x="diagnosis", data=df)
plt.show()

In [ ]:
df.hist(figsize=(14, 12))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), cmap="coolwarm")
plt.show()

In [ ]:
df.drop("diagnosis", axis=1).boxplot(figsize=(14, 6), rot=90)
plt.show()

In [ ]:
X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

In [ ]:
baseline_cv = cross_val_score(model, X_train, y_train, cv=5).mean()
model.fit(X_train, y_train)
baseline_test = accuracy_score(y_test, model.predict(X_test))
baseline_cv, baseline_test

In [ ]:
grid_params = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(model, grid_params, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

In [ ]:
grid.best_params_, grid.best_score_

In [ ]:
random_params = {"logisticregression__C": np.logspace(-3, 3, 100)}
random = RandomizedSearchCV(model, random_params, n_iter=15, cv=5, scoring="accuracy", random_state=42)
random.fit(X_train, y_train)

In [ ]:
random.best_params_, random.best_score_

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_cv = cross_val_score(rf, X_train, y_train, cv=5).mean()
rf.fit(X_train, y_train)
rf_test = accuracy_score(y_test, rf.predict(X_test))

In [ ]:
results = pd.DataFrame({
    "Model": ["Baseline Logistic Regression", "GridSearchCV", "RandomizedSearchCV", "Random Forest"],
    "CV Accuracy": [baseline_cv, grid.best_score_, random.best_score_, rf_cv],
    "Test Accuracy": [baseline_test, accuracy_score(y_test, grid.predict(X_test)), accuracy_score(y_test, random.predict(X_test)), rf_test],
    "Best Params": ["Default", str(grid.best_params_), str(random.best_params_), "n_estimators=100"]
})
results.round(4)